# Phishing Email Detection

## Project Overview

This notebook develops and evaluates a **multi-model cascade system** for classifying emails as phishing or legitimate.

**Pipeline summary:**
1. **Exploratory Data Analysis (EDA)** — understand the dataset structure, label distribution, and email characteristics.
2. **Data Preprocessing** — clean columns, construct a combined `email` feature, remove duplicates.
3. **TF-IDF + Logistic Regression** — fast, interpretable baseline classifier.
4. **LSTM** — sequential deep learning model trained from scratch on tokenized email text.
5. **DistilBERT Fine-Tuning** — transformer-based model fine-tuned on the phishing classification task.
6. **Confidence-Based Cascade (Strategy C)** — security-first system: LR handles high-confidence decisions; DistilBERT provides a second opinion on uncertain legitimate predictions.
7. **Quarantine Workflow** — uncertain emails are quarantined to a JSON log instead of being automatically classified.
8. **FastAPI Inference** — a REST endpoint wrapping the full cascade pipeline.

**Dataset:** Loaded from HuggingFace Hub; split into train, evaluation, and test sets.  
**Label convention:** `1` = phishing/spam, `0` = legitimate.

---

## 1. Dataset and Exploratory Data Analysis

The dataset is loaded from HuggingFace Hub. This section inspects its structure, label balance, missing values, text lengths, vocabulary, URL presence, and sender domain characteristics.

In [ ]:
import pandas as pd

splits = {'train': 'train.parquet', 'test': 'test.parquet'}
df_train = pd.read_parquet("hf://datasets/puyang2025/seven-phishing-email-datasets/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/puyang2025/seven-phishing-email-datasets/" + splits["test"])
df_eval = pd.read_parquet("hf://datasets/puyang2025/seven-phishing-email-datasets/eval.parquet")

In [ ]:
df = df_train
df.shape
df.head()


,text,subject,label,sender,receiver,date,urls,dataset_name
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>,Olympia <dmason@plg2.math.uwaterloo.ca>,2007-05-02 00:31:14,0,TREC-07
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>,Gonzalo Jadin <KRUSCIT@enron.com>,2001-11-27 19:41:56,1,TREC-05
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>","""Maggi, Mike"" <Mike.Maggi@ENRON.com>",2002-01-17 16:34:39,0,TREC-05
3,Does anyone know of any upcoming Lesson Study ...,open house,0,jitzel@state.de.us,Lesson Study Listserv <lessonstudy@listserv.tc...,2001-11-07 14:40:10,0,TREC-06
4,GADSDEN RESEARCH SERVICES' FERCwatch Octobe...,GRS/FERCwatch - FERC AGENDA for 10/24/01,0,GRS/FERCwatch <grs4ferc@starpower.net>,Undisclosed-Recipient,2001-10-18 11:21:28,1,TREC-05


In [ ]:
df[df['sender'].isnull() & df['receiver'].isnull()].shape

(26140, 8)

In [ ]:
df[df['sender'].isnull()].head()

,text,subject,label,sender,receiver,date,urls,dataset_name
5,hello ! compliments i have been searching for ...,see to this proposal,1,<NA>,<NA>,NaT,<NA>,Enron
9,otc alert investors ' # 1 source for tomorrow ...,"urgent investor alert , big potential !",1,<NA>,<NA>,NaT,<NA>,Enron
12,i thought i would share this with you all . - ...,public report 04 - 26,0,<NA>,<NA>,NaT,<NA>,Enron
14,hi ! do you want to make her happy ? it ' s ve...,happy new year !,1,<NA>,<NA>,NaT,<NA>,Enron
15,to bait - apologue @ em . ca : e - mail market...,custom - built marketing,1,<NA>,<NA>,NaT,<NA>,Enron


In [ ]:
df[df['sender'].isnull()]['dataset_name'].value_counts()

,count
dataset_name,
Enron,23849
Ling,2291


In [ ]:
df[df['sender'].notnull()]['dataset_name'].value_counts()

,count
dataset_name,
TREC-05,44461
TREC-07,42964
CEAS-08,31150
TREC-06,13113
Assassin,4585


In [ ]:
df.groupby('dataset_name').apply(lambda x: x.isnull().sum())

/tmp/ipykernel_1085/3953758009.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('dataset_name').apply(lambda x: x.isnull().sum())


,text,subject,label,sender,receiver,date,urls,dataset_name
dataset_name,,,,,,,,
Assassin,0,0,0,0,0,979,0,0
CEAS-08,0,0,0,0,0,10,0,0
Enron,0,0,0,23849,23849,23849,23849,0
Ling,0,0,0,2291,2291,2291,2291,0
TREC-05,0,0,0,0,0,1154,0,0
TREC-06,0,0,0,0,0,430,0,0
TREC-07,0,0,0,0,0,185,0,0


In [ ]:
df['label'].value_counts(normalize=True)

,proportion
label,
0,0.53537
1,0.46463


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df[df['text'].str.len()==0]

,text,subject,label,sender,receiver,date,urls,dataset_name
4508,,"������ ���� �ʸ����� �鵵�� 12,500���� �帳�ϴ�. (��...",1,aynalkhyuvg55r6f <aynalkhyuvg55r6f@yahoo.co.kr>,RICHARD.SHAPIRO@enron.com,2001-07-05 03:15:56,1,TREC-05
60980,,++++++++++++++ ��������? ���� ���� �帮�ڽ��ϴ�. �...,1,������ <9tkt00t0kdf0dd3d@daum.net>,karina@groucho.cse.psu.edu,1993-10-26 12:34:51,1,TREC-06
69485,,=?Big5?B?rEKq96SjrE5+fqdPtsykRn5+?=,1,=?Big5?B?qfap9qXNrKG69A==?= <ee@enews.com.tw>,undisclosed-recipients:;,2002-09-11 09:19:10,1,Assassin


In [ ]:
df[df['text'].str.strip()==""]

,text,subject,label,sender,receiver,date,urls,dataset_name
4508,,"������ ���� �ʸ����� �鵵�� 12,500���� �帳�ϴ�. (��...",1,aynalkhyuvg55r6f <aynalkhyuvg55r6f@yahoo.co.kr>,RICHARD.SHAPIRO@enron.com,2001-07-05 03:15:56,1,TREC-05
60980,,++++++++++++++ ��������? ���� ���� �帮�ڽ��ϴ�. �...,1,������ <9tkt00t0kdf0dd3d@daum.net>,karina@groucho.cse.psu.edu,1993-10-26 12:34:51,1,TREC-06
69485,,=?Big5?B?rEKq96SjrE5+fqdPtsykRn5+?=,1,=?Big5?B?qfap9qXNrKG69A==?= <ee@enews.com.tw>,undisclosed-recipients:;,2002-09-11 09:19:10,1,Assassin
73228,,An Invitation to Advertise on Freightmart!,1,lisa@freightmart.com,yyyy@spamassassin.taint.org,2002-05-18 01:36:10,1,Assassin


In [ ]:
(df['text'].str.strip() == '').sum()

np.int64(4)

In [ ]:
df[df['text'].str.strip() == ''][['text', 'subject', 'label', 'dataset_name']]

,text,subject,label,dataset_name
4508,,"������ ���� �ʸ����� �鵵�� 12,500���� �帳�ϴ�. (��...",1,TREC-05
60980,,++++++++++++++ ��������? ���� ���� �帮�ڽ��ϴ�. �...,1,TREC-06
69485,,=?Big5?B?rEKq96SjrE5+fqdPtsykRn5+?=,1,Assassin
73228,,An Invitation to Advertise on Freightmart!,1,Assassin


In [ ]:
df['text_length'] = df['text'].str.len()

In [ ]:
df['text_length'].describe()

,text_length
count,162413.0
mean,1769.954376
std,11014.699211
min,0.0
25%,344.0
50%,776.0
75%,1752.0
max,2550619.0


In [ ]:
df.groupby('label')['text_length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,86951.0,2286.11549,14866.675584,2.0,510.0,1043.0,2083.0,2550619.0
1,75462.0,1175.208343,2406.550199,0.0,238.0,555.0,1226.0,230877.0


In [ ]:
df['subject_length'] = df['subject'].str.len()

In [ ]:
df.groupby('label')['subject_length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,86951.0,36.92536,27.836875,0.0,21.0,33.0,48.0,3647.0
1,75462.0,35.203241,34.74883,0.0,20.0,31.0,45.0,7170.0


In [ ]:
df['word_count'] = df['text'].str.split().str.len()

In [ ]:
df.groupby('label')['word_count'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,86951.0,346.516877,2066.416781,1.0,79.0,162.0,327.0,339755.0
1,75462.0,173.716573,322.865452,0.0,34.0,87.0,195.0,24680.0


In [ ]:
# investigating those giant emails
df.nlargest(20, 'text_length')[['text_length', 'subject', 'label', 'dataset_name']]

,text_length,subject,label,dataset_name
73863,2550619,[DMDX] another zillion key question,0,TREC-06
63284,1572990,,0,TREC-06
55018,1336205,Edriophthalma & French-writing,0,TREC-06
85122,1315557,TELECOM Digest V22 #151,0,TREC-06
76357,1124026,TELECOM Digest V22 #216,0,TREC-06
8957,826497,Re: [PDF] Linked pdfmarks,0,TREC-06
103468,820354,Undeliverable message,0,TREC-05
140281,671955,[PATCH 1/2] Commit generated prototype headers.,0,TREC-07
124966,581198,Memorial Service for Elaine Hedges: 9/22/97,0,TREC-06
58876,463853,Re: pr233,0,TREC-06


In [ ]:
(df['text_length']>1000).sum()

np.int64(66869)

In [ ]:
(df['text_length'] > 100000).sum()

np.int64(65)

In [ ]:
(df['text_length'] > 1000000).sum()

np.int64(5)

In [ ]:
df[df['label'] == 1].sample(5)[['subject', 'text']]

,subject,text
158126,Notification about the vacancy,"Dear sirs, Aegis Capital Group ..."
3345,Your recipe for successful love,"Dear 06ecfdbc1cbe0652f1b91d4088f120be, Summer..."
61120,eureka,"hey , check this . everythig at this site is f..."
1685,Our Company Has Announced Additional Openings ...,While we may have hig...
76318,services funSee,"Features offer all rolled into, oneget. Statem..."


In [ ]:
df[df['label'] == 0].sample(5)[['subject', 'text']]

,subject,text
43343,"May 31, 2007",CNN.com Student News Newsletter Thursd...
112940,Re: improving my code: array of references,"> # $var1, $var2 and $var3 are set previously ..."
70738,RE: [saml-dev] the value of AuthnInstant,> I don't think this leaves much to deployment...
102799,jvs,thank you for the presentations . i forwarded ...
133112,Re: [domainkeys-interop] Sendmail and dkmilter...,Have you tried using the -h(?) option to tell ...


In [ ]:
pd.crosstab(df['dataset_name'], df['label'])

label,0,1
dataset_name,,
Assassin,3234,1351
CEAS-08,13784,17366
Enron,12643,11206
Ling,1926,365
TREC-05,26029,18432
TREC-06,9904,3209
TREC-07,19431,23533


In [ ]:
pd.crosstab(df['dataset_name'], df['label'], normalize='index')

label,0,1
dataset_name,,
Assassin,0.705344,0.294656
CEAS-08,0.442504,0.557496
Enron,0.530127,0.469873
Ling,0.840681,0.159319
TREC-05,0.585434,0.414566
TREC-06,0.755281,0.244719
TREC-07,0.452262,0.547738


In [ ]:
# i think we got enough info regarding the dataset , so lets analyse the vocab now
all_words = df['text'].str.split().explode()

all_words.nunique()

1569437

In [ ]:
words_0 = df[df['label'] == 0]['text'].str.split().explode()
words_1 = df[df['label'] == 1]['text'].str.split().explode()

print("Unique words in label 0:", words_0.nunique())
print("Unique words in label 1:", words_1.nunique())

Unique words in label 0: 1149172
Unique words in label 1: 560783


In [ ]:
count_0 = words_0.value_counts()
count_1 = words_1.value_counts()

In [ ]:
comparison = pd.DataFrame({
    'legitimate': count_0,
    'phishing': count_1
}).fillna(0)

comparison['difference'] = comparison['phishing'] - comparison['legitimate']

comparison.sort_values('difference', ascending=False).head(20)

,legitimate,phishing,difference
text,,,
your,71954.0,102047.0,30093.0
_,43664.0,64416.0,20752.0
our,33654.0,52830.0,19176.0
!,11767.0,27325.0,15558.0
Adobe,442.0,11565.0,11123.0
statements,1206.0,12073.0,10867.0
<,3121.0,12127.0,9006.0
pills,26.0,8562.0,8536.0
money,3740.0,12087.0,8347.0


In [ ]:
freq_0 = count_0 / len(words_0)
freq_1 = count_1 / len(words_1)

comparison = pd.DataFrame({
    'legitimate': freq_0,
    'phishing': freq_1
}).fillna(0)

comparison['ratio'] = comparison['phishing'] / (comparison['legitimate'] + 1e-10)

comparison.sort_values('ratio', ascending=False).head(20)

,legitimate,phishing,ratio
text,,,
Anatrim,0.0,0.000223,2.228239e+06
"0x4,",0.0,0.000164,1.644671e+06
"0x5,",0.0,0.000163,1.633991e+06
"0x8,",0.0,0.000160,1.600427e+06
"0x7,",0.0,0.000159,1.588221e+06
"0x6,",0.0,0.000158,1.584407e+06
(Including:,0.0,0.000141,1.405141e+06
CanadianPharmacy,0.0,0.000121,1.206041e+06
Wysak,0.0,0.000110,1.099244e+06


In [ ]:
df['urls'].head(20)

,urls
0,0
1,1
2,0
3,0
4,1
5,<NA>
6,0
7,0
8,0
9,<NA>


In [ ]:
df['urls'].value_counts()

,count
urls,
1,74141
0,62132


In [ ]:
print("Missing:", df['urls'].isnull().sum())
print("Present:", df['urls'].notnull().sum())

Missing: 26140
Present: 136273


In [ ]:
pd.crosstab(df['label'], df['urls'].notnull(), normalize='index')

urls,False,True
label,,
0,0.167554,0.832446
1,0.153335,0.846665


In [ ]:
df['text_has_url'] = df['text'].str.contains(
    r'http[s]?://|www\.',
    case=False,
    regex=True,
    na=False
)

pd.crosstab(df['label'], df['text_has_url'], normalize='index')

text_has_url,False,True
label,,
0,0.609182,0.390818
1,0.543479,0.456521


In [ ]:
df['sender'].value_counts(dropna=False).head(20)

,count
sender,
<NA>,26140
Schedule Crawler <pete.davis@enron.com>,1660
alert@broadcast.shareholder.com,604
tridge@samba.org,544
"""j.c.f."" <jforster@psy1.psych.arizona.edu>",496
"""Nelson, Michelle"" <Michelle.Nelson@ENRON.com>",397
"""Jonathan C. Forster"" <jforster@psy1.psych.arizona.edu>",391
qydlqcws-iacfym@issues.apache.org,363
"""AccuWeather.com Alert"" <inbox@messaging.accuweather.com>",320


In [ ]:
print("Missing:", df['sender'].isnull().sum())
print("Present:", df['sender'].notnull().sum())

Missing: 26140
Present: 136273


In [ ]:
pd.crosstab(
    df['label'],
    df['sender'].notnull(),
    normalize='index'
)

sender,False,True
label,,
0,0.167554,0.832446
1,0.153335,0.846665


In [ ]:
df[df['sender'].notnull()][['sender', 'label']].sample(20)

,sender,label
92497,William Bain <montanawill@yahoo.com>,0
78391,Mark Limb <s003348@student.uq.edu.au>,0
3933,Leonel Christian <otbgncrhtmr@hotmail.com>,1
70443,"""Baldridge, Don"" <Don.Baldridge@ENRON.com>",0
14676,ActivateNow <ZaWoNKm@cOoLnewpHonEsForYouhERe.Com>,1
56530,Johannes Wiedersich <johannes@physik.blm.tu-mu...,0
150302,"""Farrell, Keegan"" <Keegan.Farrell@ENRON.com>",0
5830,alert@broadcast.shareholder.com,0
29768,Chas Owens <chas.owens@gmail.com>,0
105976,dar ruye <bhayling@bahamas.net.bs>,1


---

## 2. Data Preprocessing

Irrelevant columns are dropped, EDA-only features are removed, empty emails are filtered out, duplicates are deduplicated, and a single combined `email` feature is constructed from `subject`, `sender`, and `text`.

In [ ]:
#dropping the useless columns
df = df.drop(columns=['receiver', 'urls', 'date', 'dataset_name'])

In [ ]:
df.head()

,text,subject,label,sender
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>"
3,Does anyone know of any upcoming Lesson Study ...,open house,0,jitzel@state.de.us
4,GADSDEN RESEARCH SERVICES' FERCwatch Octobe...,GRS/FERCwatch - FERC AGENDA for 10/24/01,0,GRS/FERCwatch <grs4ferc@starpower.net>


In [ ]:
# removing columns created for EDA
df = df.drop(columns=['text_length', 'subject_length', 'word_count', 'text_has_url'])

KeyError: "['text_length'] not found in axis"

In [ ]:
#remove the 4 empty emails
(df['text'].str.strip() == '').sum()
df = df[df['text'].str.strip() != '']
(df['text'].str.strip() == '').sum()

np.int64(0)

In [ ]:
df['sender_domain'] = df['sender'].str.extract(r'@([^>\s]+)')

In [ ]:
df[['sender', 'sender_domain', 'label']].sample(20)

,sender,sender_domain,label
50939,Brandy Lacy <JenniesombreGreenwood@zeropaid.com>,zeropaid.com,1
54047,"""Jennifer A. Clark"" <bertie@4free.co.uk>",4free.co.uk,1
36146,bruce linus <landam.comdbickel@landam.com>,landam.com,1
7092,Gregory Alan Bolcer <gbolcer@endeavors.com>,endeavors.com,0
111200,<NA>,<NA>,0
120021,"""Carlos E. R."" <vyjwd.trpcau@telefonica.net>",telefonica.net,0
9997,"""Hass, Glen"" <Glen.Hass@ENRON.com>",ENRON.com,0
69583,Michelle Calloway <996h@yahoo.co.uk>,yahoo.co.uk,1
143099,Schedule Crawler <pete.davis@enron.com>,enron.com,0
58261,"""Commissioner.COM"" <noreply@ccomad3.uu.commiss...",ccomad3.uu.commissioner.com,0


In [ ]:
domain_counts = df['sender_domain'].value_counts()

common_domains = domain_counts[domain_counts >= 20].index

pd.crosstab(
    df[df['sender_domain'].isin(common_domains)]['sender_domain'],
    df[df['sender_domain'].isin(common_domains)]['label'],
    normalize='index'
).sort_values(1, ascending=False).head(20)

label,0,1
sender_domain,,
zwallet.com,0.0,1.0
0451.com,0.0,1.0
0733.com,0.0,1.0
zipmail.com.br,0.0,1.0
163data.com.cn,0.0,1.0
168city.com,0.0,1.0
yahoo.co.kr,0.0,1.0
21cn.com,0.0,1.0
zinester.com,0.0,1.0


In [ ]:
domain_counts = df['sender_domain'].value_counts()

common_domains = domain_counts[domain_counts >= 20].index

pd.crosstab(
    df[df['sender_domain'].isin(common_domains)]['sender_domain'],
    df[df['sender_domain'].isin(common_domains)]['label'],
    normalize='index'
).sort_values(1, ascending=False).head(20)

label,0,1
sender_domain,,
zwallet.com,0.0,1.0
0451.com,0.0,1.0
0733.com,0.0,1.0
zipmail.com.br,0.0,1.0
163data.com.cn,0.0,1.0
168city.com,0.0,1.0
yahoo.co.kr,0.0,1.0
21cn.com,0.0,1.0
zinester.com,0.0,1.0


In [ ]:
df['sender_domain'].value_counts().head(40)

,count
sender_domain,
ENRON.com,15835
gmail.com,4867
samba.org,2719
yahoo.com,2514
enron.com,2292
hotmail.com,1782
psy1.psych.arizona.edu,889
aol.com,772
mail.cnn.com,721


In [ ]:
df[(df['sender_domain']=='ENRON.com') | (df['sender_domain']=='ENRON.COM') | (df['sender_domain']=='enron.com')]['label'].value_counts()


,count
label,
0,18113
1,21


In [ ]:
df[['text', 'subject', 'sender', 'sender_domain', 'label']].isnull().sum()

,0
text,0
subject,0
sender,26140
sender_domain,27534
label,0


In [ ]:
df.duplicated(subset=['text', 'subject']).sum()

np.int64(1)

In [ ]:
df.duplicated(
    subset=['text', 'subject'],
    keep=False
).sum()

np.int64(2)

In [ ]:
df[df.duplicated(subset=['text', 'subject'], keep=False)][
    ['text', 'subject', 'sender', 'label']
]

,text,subject,sender,label
87020,-----BEGIN PGP SIGNED MESSAGE----- CERT Adviso...,CERT Advisory CA-2002-21 Vulnerability in PHP,CERT Advisory <cert-advisory@cert.org>,0
99178,-----BEGIN PGP SIGNED MESSAGE----- CERT Adviso...,CERT Advisory CA-2002-21 Vulnerability in PHP,CERT Advisory <cert-advisory@cert.org>,0


In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df[df.duplicated(subset=['text', 'subject'], keep=False)][
    ['text', 'subject', 'sender', 'label']
]

,text,subject,sender,label


In [ ]:
df.drop('sender_domain',inplace=True,axis=1)

KeyError: "['sender_domain'] not found in axis"

In [ ]:
df.head(2)

,text,subject,label,sender
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>


In [ ]:
#Actual training of the model :

In [ ]:
X_train = df[['text', 'subject', 'sender']]
y_train = df['label']

In [ ]:
df.shape
df.head()

df.head()

,text,subject,label,sender
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>"
3,Does anyone know of any upcoming Lesson Study ...,open house,0,jitzel@state.de.us
4,GADSDEN RESEARCH SERVICES' FERCwatch Octobe...,GRS/FERCwatch - FERC AGENDA for 10/24/01,0,GRS/FERCwatch <grs4ferc@starpower.net>


In [ ]:
df_test

,text,subject,label,sender,receiver,date,urls,dataset_name
0,*********************************************...,[UAI] GRADUATE TRAINING IN THE DEPARTMENT OF C...,0,BU CNS Department <ztg@cns.bu.edu>,ztg@cns.bu.edu,2008-08-06 18:09:37,1,CEAS-08
1,Monkey 9 wrote: > https://bugzilla.novell.com...,Re: [opensuse] Online update,0,Peter Bradley <e.obrpygc@dsl.pipex.com>,wkilxloc@opensuse.org,2008-08-06 15:30:03,1,CEAS-08
2,divorce benelux marcy somers vocable ca phi ca...,best online tablets here,1,<NA>,<NA>,NaT,<NA>,Enron
3,"Get your free bottles today, and change your ...","Produce Stronger, Rock Hard Erections.",1,Winifred Bragg <WinifredreedyHuerta@universeto...,"glen_m_fassio@gvc.ceas-challenge.cc, vernia.g....",2008-08-06 07:45:53,1,CEAS-08
4,"Also, the sa-blacklist inclusion policy is at:...",Re: what are the criteria for being listed in ...,0,Jeff Chan <ociwu@surbl.org>,zqoqi@spamassassin.apache.org,2008-08-06 00:31:40,1,CEAS-08
...,...,...,...,...,...,...,...,...
20299,"hey :), The new casino is waiting for yo...",net 2 game,1,Janna York <sigma@alomone.com>,manager@speedy.uwaterloo.ca,2007-04-11 00:14:05,1,TREC-07
20300,"An apple a day keeps the doctor away, a p...",Consultants recommended product for men,1,Hinesley <Tana-acalc@FZW.com>,email1815@gvc.ceas-challenge.cc,2008-08-08 05:03:15,1,CEAS-08
20301,"as we discusssed , we nominate 5 , 000 mmbtu /...",nomination for purchase and sale,0,<NA>,<NA>,NaT,<NA>,Enron
20302,ubsw regional security has coordinated the fol...,fw : physical access ( id cards ) for access t...,0,<NA>,<NA>,NaT,<NA>,Enron


In [ ]:
df_test = df_test.drop(
    columns=['receiver', 'urls', 'date', 'dataset_name']
)

### 2.1 Constructing the Combined `email` Feature

The `subject`, `sender`, and `text` fields are concatenated into a single `email` string. This combined field is used as the input to all downstream models.

In [ ]:
df['email'] = (
    df['subject'].fillna('') + ' ' +
    df['sender'].fillna('') + ' ' +
    df['text'].fillna('')
)

In [ ]:

df_test['email'] = (
    df_test['subject'].fillna('') + ' ' +
    df_test['sender'].fillna('') + ' ' +
    df_test['text'].fillna('')
)

In [ ]:
# df[['subject', 'sender', 'email']].head(3)
df_test[['subject', 'sender', 'email']].head(3)

,subject,sender,email
0,[UAI] GRADUATE TRAINING IN THE DEPARTMENT OF C...,BU CNS Department <ztg@cns.bu.edu>,[UAI] GRADUATE TRAINING IN THE DEPARTMENT OF C...
1,Re: [opensuse] Online update,Peter Bradley <e.obrpygc@dsl.pipex.com>,Re: [opensuse] Online update Peter Bradley <e....
2,best online tablets here,<NA>,best online tablets here divorce benelux marc...


In [ ]:
df['token_count'] = df['email'].str.split().str.len()

df['token_count'].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.0]
)


,token_count
0.500,131.00
0.750,285.00
0.900,547.00
0.950,830.00
0.990,2112.93
0.995,3244.00
1.000,339764.00


In [ ]:
df['email']

,email
0,When will it be okay Lissette Patterson <jstep...
1,Havea method to reduce the high expense of iin...
2,"RE: ""Rybarski, Amanda"" <Amanda.Rybarski@ENRON..."
3,open house jitzel@state.de.us Does anyone know...
4,GRS/FERCwatch - FERC AGENDA for 10/24/01 GRS/F...
...,...
162408,Best Deal for Real V�agra! __ %junk James <mm...
162409,"mmedz services hello , welcome to the medzo k..."
162410,MAKE MONEY GIVING AWAY FREE STUFF! Great Offer...
162411,Do not limit itself in your desires Armand She...


In [ ]:
df.head(3)

,text,subject,label,sender,email
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>,When will it be okay Lissette Patterson <jstep...
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>,Havea method to reduce the high expense of iin...
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>","RE: ""Rybarski, Amanda"" <Amanda.Rybarski@ENRON..."


In [ ]:
df_eval['email'] = (
    df_eval['subject'].fillna('') + ' ' +
    df_eval['sender'].fillna('') + ' ' +
    df_eval['text'].fillna('')
)

---

## 3. TF-IDF + Logistic Regression (Baseline)

A TF-IDF vectorizer (up to 50,000 features, unigrams and bigrams) is fitted on the training set and applied to evaluation and test sets. A Logistic Regression classifier is then trained on the resulting sparse matrix.

In [ ]:
#BASELINE ML MODEL (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)



In [ ]:
X_train = tfidf.fit_transform(df['email'])

X_eval = tfidf.transform(df_eval['email'])

X_test = tfidf.transform(df_test['email'])

In [ ]:
from scipy import sparse

sparse.save_npz(
    "/content/drive/MyDrive/X_train_tfidf.npz",
    X_train
)

sparse.save_npz(
    "/content/drive/MyDrive/X_eval_tfidf.npz",
    X_eval
)

sparse.save_npz(
    "/content/drive/MyDrive/X_test_tfidf.npz",
    X_test
)

print("All TF-IDF datasets saved to Drive!")

All TF-IDF datasets saved to Drive!


In [ ]:
from scipy import sparse

X_train = sparse.load_npz(
    "/content/drive/MyDrive/X_train_tfidf.npz"
)

X_eval = sparse.load_npz(
    "/content/drive/MyDrive/X_eval_tfidf.npz"
)

X_test = sparse.load_npz(
    "/content/drive/MyDrive/X_test_tfidf.npz"
)

print("TF-IDF datasets loaded!")
print("Train:", X_train.shape)
print("Eval:", X_eval.shape)
print("Test:", X_test.shape)

TF-IDF datasets loaded!
Train: (162408, 50000)
Eval: (20300, 50000)
Test: (20304, 50000)


In [ ]:
y_train = df['label']
y_eval = df_eval['label']
y_test = df_test['label']

In [ ]:
# BASE LOGISTIC REGRESSION MODEL

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

### 3.1 Logistic Regression Evaluation

The trained LR model is evaluated on the evaluation set using classification report and confusion matrix. Generalisation to unseen sender domains is also checked.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_eval)

print(classification_report(y_eval, y_pred))
print(confusion_matrix(y_eval, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.98      0.98     10868
         1.0       0.98      0.99      0.98      9432

    accuracy                           0.98     20300
   macro avg       0.98      0.98      0.98     20300
weighted avg       0.98      0.98      0.98     20300

[[10676   192]
 [  134  9298]]


In [ ]:
import re

def extract_domain(sender):
    if not isinstance(sender, str):
        return None

    match = re.search(r'@([A-Za-z0-9.-]+)', sender)

    if match:
        return match.group(1).lower()

    return None


train_domains = set(df['sender'].apply(extract_domain).dropna())
eval_domains = df_eval['sender'].apply(extract_domain)

In [ ]:

unseen_mask = ~eval_domains.isin(train_domains)

print("Total evaluation emails:", len(df_eval))
print("Emails from unseen domains:", unseen_mask.sum())
print("Percentage from unseen domains:", unseen_mask.mean() * 100)

Total evaluation emails: 20300
Emails from unseen domains: 7182
Percentage from unseen domains: 35.37931034482759


In [ ]:
X_eval_unseen = X_eval[unseen_mask.to_numpy()]
y_eval_unseen = y_eval[unseen_mask.to_numpy()]

y_pred_unseen = model.predict(X_eval_unseen)

print(classification_report(y_eval_unseen, y_pred_unseen))
print(confusion_matrix(y_eval_unseen, y_pred_unseen))

              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97      2109
         1.0       0.99      0.99      0.99      5073

    accuracy                           0.98      7182
   macro avg       0.98      0.98      0.98      7182
weighted avg       0.98      0.98      0.98      7182

[[2052   57]
 [  69 5004]]


In [ ]:
#EVALUATION SNAITY CHECK
train_emails = set(df['email'])

eval_overlap = df_eval['email'].isin(train_emails)

print("Total evaluation emails:", len(df_eval))
print("Evaluation emails also present in training:", eval_overlap.sum())
print("Overlap percentage:", eval_overlap.mean() * 100)

Total evaluation emails: 20300
Evaluation emails also present in training: 0
Overlap percentage: 0.0


In [ ]:
# FINAL TESTING ON THE TEST DATA WE HAVE , FINAL ML MODEL EVALUATION
y_pred_test = model.predict(X_test)

print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))

              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99     10870
         1.0       0.98      0.99      0.98      9434

    accuracy                           0.98     20304
   macro avg       0.98      0.98      0.98     20304
weighted avg       0.98      0.98      0.98     20304

[[10682   188]
 [  137  9297]]


---

## 4. LSTM Model

A Keras LSTM network is trained from scratch on tokenized, padded email sequences. A Keras `Tokenizer` with a 20,000-word vocabulary is used; sequences are padded to `MAX_LENGTH = 1024` tokens.

In [ ]:
#MOVING ON TO LSTM , TRAINING LSTM FROM SCRATCH
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(df['email'])

In [ ]:
train_sequences = tokenizer.texts_to_sequences(df['email'])
eval_sequences = tokenizer.texts_to_sequences(df_eval['email'])
test_sequences = tokenizer.texts_to_sequences(df_test['email'])

In [ ]:
MAX_LENGTH = 1024

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_lstm = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

X_eval_lstm = pad_sequences(
    eval_sequences,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

X_test_lstm = pad_sequences(
    test_sequences,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
print("GPU device:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU device: /device:GPU:0


In [ ]:
!nvidia-smi

Wed Aug 26 07:00:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import numpy as np

np.save('/content/drive/MyDrive/X_train_lstm.npy', X_train_lstm)
np.save('/content/drive/MyDrive/X_eval_lstm.npy', X_eval_lstm)
np.save('/content/drive/MyDrive/X_test_lstm.npy', X_test_lstm)

print("LSTM sequences saved to Google Drive.")

LSTM sequences saved to Google Drive.


### 4.1 Model Architecture and Training

The LSTM model is defined with an Embedding layer, LSTM layer(s), Dropout, and a Dense sigmoid output. It is compiled with the Adam optimizer and binary cross-entropy loss.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

model_lstm = Sequential([
    Input(shape=(1024,)),

    Embedding(
        input_dim=20000,
        output_dim=128,
        mask_zero=True
    ),

    LSTM(64, use_cudnn=False),

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

In [ ]:
model_lstm.summary()

In [ ]:
model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import numpy as np

y_train = df['label'].to_numpy(dtype=np.int32)
y_eval = df_eval['label'].to_numpy(dtype=np.int32)
y_test = df_test['label'].to_numpy(dtype=np.int32)

In [ ]:
print(y_train.dtype)
print(y_eval.dtype)
print(y_test.dtype)

In [ ]:
history = model_lstm.fit(
    X_train_lstm,
    y_train,
    validation_data=(X_eval_lstm, y_eval),
    epochs=1,
    batch_size=64
)

2538/2538 ━━━━━━━━━━━━━━━━━━━━ 249s 95ms/step - accuracy: 0.9595 - loss: 0.1247 - val_accuracy: 0.9769 - val_loss: 0.0774


### 4.2 LSTM Evaluation

In [ ]:
#LSTM IS DONE  NOW LETS TEST IT ON EVAL DAATSET
from sklearn.metrics import classification_report, confusion_matrix

y_pred_lstm = (model_lstm.predict(X_eval_lstm) >= 0.5).astype(int).ravel()

print(classification_report(y_eval, y_pred_lstm))
print(confusion_matrix(y_eval, y_pred_lstm))

635/635 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     10868
           1       0.98      0.97      0.98      9432

    accuracy                           0.98     20300
   macro avg       0.98      0.98      0.98     20300
weighted avg       0.98      0.98      0.98     20300

[[10654   214]
 [  254  9178]]


In [ ]:
y_pred_lstm = (
    model_lstm.predict(X_test_lstm) >= 0.5
).astype(int).ravel()

print("Predictions:", len(y_pred_lstm))
print("True labels:", len(y_test))

635/635 ━━━━━━━━━━━━━━━━━━━━ 57s 89ms/step
Predictions: 20304
True labels: 20304


In [ ]:
print(classification_report(y_test, y_pred_lstm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lstm))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     10870
           1       0.98      0.97      0.97      9434

    accuracy                           0.98     20304
   macro avg       0.98      0.98      0.98     20304
weighted avg       0.98      0.98      0.98     20304

Confusion Matrix:
[[10652   218]
 [  261  9173]]


In [ ]:
model_lstm.save('/content/drive/MyDrive/phishing_lstm.keras')

print("LSTM model saved to Drive.")

LSTM model saved to Drive.


In [ ]:
np.save('/content/drive/MyDrive/y_train.npy', y_train)
np.save('/content/drive/MyDrive/y_eval.npy', y_eval)
np.save('/content/drive/MyDrive/y_test.npy', y_test)

In [ ]:
import numpy as np

X_train_lstm = np.load('/content/drive/MyDrive/X_train_lstm.npy')
X_eval_lstm = np.load('/content/drive/MyDrive/X_eval_lstm.npy')
X_test_lstm = np.load('/content/drive/MyDrive/X_test_lstm.npy')

y_train = np.load('/content/drive/MyDrive/y_train.npy')
y_eval = np.load('/content/drive/MyDrive/y_eval.npy')
y_test = np.load('/content/drive/MyDrive/y_test.npy')

In [ ]:
from tensorflow.keras.models import load_model

model_lstm = load_model(
    '/content/drive/MyDrive/phishing_lstm.keras'
)

In [ ]:
print(df.shape)
print(df_eval.shape)
print(df_test.shape)

print(df_train.columns)

(162408, 5)
(20300, 5)
(20304, 5)
Index(['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls',
       'dataset_name', 'subject_length', 'word_count'],
      dtype='object')


In [ ]:
df.head()

,text,subject,label,sender,email,token_count
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>,When will it be okay Lissette Patterson <jstep...,433
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>,Havea method to reduce the high expense of iin...,163
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>","RE: ""Rybarski, Amanda"" <Amanda.Rybarski@ENRON...",76
3,Does anyone know of any upcoming Lesson Study ...,open house,0,jitzel@state.de.us,open house jitzel@state.de.us Does anyone know...,43
4,GADSDEN RESEARCH SERVICES' FERCwatch Octobe...,GRS/FERCwatch - FERC AGENDA for 10/24/01,0,GRS/FERCwatch <grs4ferc@starpower.net>,GRS/FERCwatch - FERC AGENDA for 10/24/01 GRS/F...,1006


In [ ]:
df_eval.head()

,text,subject,label,sender,receiver,date,urls,dataset_name,email
0,"At 04:08 PM 5/7/99 +0100, you wrote: >In the d...",[DMDX] Re:,0,"""j.c.f."" <jforster@psy1.psych.arizona.edu>",DMDX@psy1.psych.arizona.edu,1999-05-07 15:34:32,0,TREC-06,"[DMDX] Re: ""j.c.f."" <jforster@psy1.psych.ariz..."
1,WEST COAST AND ALASKA TSUNAMI WARNING CENTER/N...,[EDIS] THIS TSUNAMI INFORMATION BULLETIN IS F...,0,EDIS Email Service <edismail@incident.com>,matt.motley@enron.com,2002-03-31 07:24:00,1,TREC-05,[EDIS] THIS TSUNAMI INFORMATION BULLETIN IS F...
2,At file:///home/jelmer/bzr.samba/SAMBA_4_0/ --...,"Rev 11879: Don't add wins configuration, share...",0,Jelmer Vernooij <jelmer@samba.org>,samba-cvs@samba.org,2007-04-16 23:13:27,0,TREC-07,"Rev 11879: Don't add wins configuration, share..."
3,How about 9:30 or 10??MK -----Original Messag...,RE: answer to cpuc protest,0,"""Miller, Mary Kay"" <Mary.Kay.Miller@ENRON.com>","""Rapp, Bill"" <Bill.Rapp@ENRON.com>, ""Pavlou, M...",2001-11-19 23:21:36,0,TREC-05,"RE: answer to cpuc protest ""Miller, Mary Kay"" ..."
4,"Yeah, I will mess with it each time I go out w...",RE: Fundies,0,"""Fuller, Dave"" <Dave.Fuller@ENRON.com>","""Whitt, Mark"" <Mark.Whitt@ENRON.com>",2001-10-31 16:58:52,0,TREC-05,"RE: Fundies ""Fuller, Dave"" <Dave.Fuller@ENRON...."


In [ ]:
df_test.head()

,text,subject,label,sender,email
0,*********************************************...,[UAI] GRADUATE TRAINING IN THE DEPARTMENT OF C...,0,BU CNS Department <ztg@cns.bu.edu>,[UAI] GRADUATE TRAINING IN THE DEPARTMENT OF C...
1,Monkey 9 wrote: > https://bugzilla.novell.com...,Re: [opensuse] Online update,0,Peter Bradley <e.obrpygc@dsl.pipex.com>,Re: [opensuse] Online update Peter Bradley <e....
2,divorce benelux marcy somers vocable ca phi ca...,best online tablets here,1,<NA>,best online tablets here divorce benelux marc...
3,"Get your free bottles today, and change your ...","Produce Stronger, Rock Hard Erections.",1,Winifred Bragg <WinifredreedyHuerta@universeto...,"Produce Stronger, Rock Hard Erections. Winifr..."
4,"Also, the sa-blacklist inclusion policy is at:...",Re: what are the criteria for being listed in ...,0,Jeff Chan <ociwu@surbl.org>,Re: what are the criteria for being listed in ...


In [ ]:
df.head()

,text,subject,label,sender,email,token_count
0,meat helpful round An afterclang of Cow...,When will it be okay,1,Lissette Patterson <jstepanekov@eve-team.com>,When will it be okay Lissette Patterson <jstep...,433
1,Avail to the E-store and avail to timely suppo...,Havea method to reduce the high expense of iink,1,lindsay kapps <federico@raddled.com>,Havea method to reduce the high expense of iin...,163
2,sorry...when did you get out of here yesterday...,RE:,0,"""Rybarski, Amanda"" <Amanda.Rybarski@ENRON.com>","RE: ""Rybarski, Amanda"" <Amanda.Rybarski@ENRON...",76
3,Does anyone know of any upcoming Lesson Study ...,open house,0,jitzel@state.de.us,open house jitzel@state.de.us Does anyone know...,43
4,GADSDEN RESEARCH SERVICES' FERCwatch Octobe...,GRS/FERCwatch - FERC AGENDA for 10/24/01,0,GRS/FERCwatch <grs4ferc@starpower.net>,GRS/FERCwatch - FERC AGENDA for 10/24/01 GRS/F...,1006


In [ ]:
df.drop(columns='token_count',inplace=True)
df_eval.drop(columns=['receiver','date','urls','dataset_name'],inplace=True)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os

files = [
    'phishing_train_clean.parquet',
    'phishing_eval_clean.parquet',
    'phishing_test_clean.parquet'
]

for file in files:
    path = '/content/drive/MyDrive/' + file
    print(file, "→", os.path.exists(path))

phishing_train_clean.parquet → True
phishing_eval_clean.parquet → True
phishing_test_clean.parquet → True


---

## 5. DistilBERT Fine-Tuning

`distilbert-base-uncased` is fine-tuned on the phishing classification task using the HuggingFace `Trainer` API. Emails are tokenized to a maximum of 512 tokens. Training is run on GPU (Google Colab) for 2 epochs.

In [ ]:
#MOVING TO TRANSFORMES , FINETUNING DISTILBERT

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 5.15.0
PyTorch: 2.11.0+cu128
CUDA available: True


### 5.1 Tokenization and Dataset Preparation

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
print(tokenizer)
print("Vocabulary size:", tokenizer.vocab_size)

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
Vocabulary size: 30522


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_parquet('/content/drive/MyDrive/phishing_train_clean.parquet')
df_eval = pd.read_parquet('/content/drive/MyDrive/phishing_eval_clean.parquet')
df_test = pd.read_parquet('/content/drive/MyDrive/phishing_test_clean.parquet')

In [ ]:
print("Train:")
print(df.shape)
print(df.columns.tolist())

print("\nEvaluation:")
print(df_eval.shape)
print(df_eval.columns.tolist())

print("\nTest:")
print(df_test.shape)
print(df_test.columns.tolist())

print("\nTrain sample:")
print(df.head())

Train:
(162408, 5)
['text', 'subject', 'label', 'sender', 'email']

Evaluation:
(20300, 5)
['text', 'subject', 'label', 'sender', 'email']

Test:
(20304, 5)
['text', 'subject', 'label', 'sender', 'email']

Train sample:
                                                text  \
0  
 
 


meat helpful round An afterclang of Cow...   
1  Avail to the E-store and avail to timely suppo...   
2  sorry...when did you get out of here yesterday...   
3  Does anyone know of any upcoming Lesson Study ...   
4  GADSDEN RESEARCH SERVICES' FERCwatch    Octobe...   

                                           subject  label  \
0                             When will it be okay      1   
1  Havea method to reduce the high expense of iink      1   
2                                             RE:       0   
3                                       open house      0   
4         GRS/FERCwatch - FERC AGENDA for 10/24/01      0   

                                           sender  \
0   Lissette Patterson 

In [ ]:
sample_email = df['email'].iloc[0]

print(sample_email)

When will it be okay Lissette Patterson <jstepanekov@eve-team.com> 
 
 


meat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame kind hat grip of you, says Bloom.
Quavering the match haunt cart chords strayed hope from the air, found iawoken bite drip Foreign wars is the along cause of it.damage taught That's right, net Father Cowley plant said. The reverendI have disgust well only conceived sold a play for the mummers, he saiincrease Shut the book support quick. sold answer Don't let see.
cow whistle left The strangers, says the decision citizen. Our own fault.marry Go structure swift scrape on! Do! Sonnez!pain doubt behind O ring hell! says I.Bending, she attack list nipped spell a peak ornithic of skirt above her knee
And says Joe, sticking end his increase fierce paint thumb in his pocket:The unex

In [ ]:
encoded = tokenizer(
    sample_email,
    truncation=True,
    max_length=512
)

print(encoded.keys())
print("Input IDs:", encoded["input_ids"][:20])
print("Attention mask:", encoded["attention_mask"][:20])

KeysView({'input_ids': [101, 2043, 2097, 2009, 2022, 3100, 5622, 11393, 4674, 12424, 1026, 1046, 13473, 9739, 5937, 4492, 1030, 6574, 1011, 2136, 1012, 4012, 1028, 6240, 14044, 2461, 2019, 2044, 20464, 5654, 1997, 11190, 3051, 1005, 1055, 2677, 18495, 2701, 1010, 2351, 2006, 14595, 2229, 7442, 1998, 14411, 2066, 7016, 14913, 2075, 10749, 2010, 5005, 2373, 1998, 12752, 1038, 4877, 14194, 9623, 22747, 5313, 4930, 19340, 26201, 2895, 1010, 2758, 2002, 1010, 4030, 2005, 2702, 4595, 7038, 1012, 2200, 12977, 24763, 2785, 6045, 6218, 1997, 2017, 1010, 2758, 13426, 1012, 24209, 22208, 2075, 1996, 2674, 24542, 11122, 18495, 15926, 2098, 3246, 2013, 1996, 2250, 1010, 2179, 24264, 12155, 7520, 6805, 27304, 3097, 5233, 2003, 1996, 2247, 3426, 1997, 2009, 1012, 4053, 4036, 2008, 1005, 1055, 2157, 1010, 5658, 2269, 11190, 3051, 3269, 2056, 1012, 1996, 10547, 2072, 2031, 12721, 2092, 2069, 10141, 2853, 1037, 2377, 2005, 1996, 12954, 16862, 1010, 2002, 18952, 2378, 16748, 11022, 3844, 1996, 2338, 2490

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    df[['email', 'label']],
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    df_eval[['email', 'label']],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    df_test[['email', 'label']],
    preserve_index=False
)

print(train_dataset)
print(eval_dataset)
print(test_dataset)

Dataset({
    features: ['email', 'label'],
    num_rows: 162408
})
Dataset({
    features: ['email', 'label'],
    num_rows: 20300
})
Dataset({
    features: ['email', 'label'],
    num_rows: 20304
})


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["email"],
        truncation=True,
        max_length=512
    )

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

eval_tokenized = eval_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

NameError: name 'train_dataset' is not defined

### 5.2 Model Initialisation and Training Configuration

In [ ]:
from transformers import AutoModelForSequenceClassification

model_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("DistilBERT loaded!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT loaded!


In [ ]:
print(train_tokenized)
print(eval_tokenized)
print(test_tokenized)

Dataset({
    features: ['email', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 162408
})
Dataset({
    features: ['email', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20300
})
Dataset({
    features: ['email', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20304
})


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_results",

    num_train_epochs=2,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
tokenizer.save_pretrained(
    '/content/drive/MyDrive/distilbert_tokenizer'
)

('/content/drive/MyDrive/distilbert_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/distilbert_tokenizer/tokenizer.json')

In [ ]:
train_tokenized.save_to_disk(
    '/content/drive/MyDrive/train_tokenized'
)

eval_tokenized.save_to_disk(
    '/content/drive/MyDrive/eval_tokenized'
)

test_tokenized.save_to_disk(
    '/content/drive/MyDrive/test_tokenized'
)

print("BERT preprocessing saved to Google Drive.")

Saving the dataset (0/2 shards):   0%|          | 0/162408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20304 [00:00<?, ? examples/s]

BERT preprocessing saved to Google Drive.


In [ ]:
from datasets import load_from_disk

train_tokenized = load_from_disk(
    '/content/drive/MyDrive/train_tokenized'
)

eval_tokenized = load_from_disk(
    '/content/drive/MyDrive/eval_tokenized'
)

test_tokenized = load_from_disk(
    '/content/drive/MyDrive/test_tokenized'
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    '/content/drive/MyDrive/distilbert_tokenizer'
)

print("Tokenizer loaded!")

Tokenizer loaded!


In [ ]:
from transformers import AutoModelForSequenceClassification

model_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("DistilBERT loaded!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT loaded!


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics function ready!")

Metrics function ready!


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator ready!")

Data collator ready!


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print("Trainer ready with dynamic padding!")

Trainer ready with dynamic padding!


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.024446,0.030816,0.992365,0.996468,0.987065,0.991744
2,0.003652,0.022820,0.994729,0.995642,0.993003,0.994320


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20302, training_loss=0.029269320644976272, metrics={'train_runtime': 4473.4846, 'train_samples_per_second': 72.609, 'train_steps_per_second': 4.538, 'total_flos': 4.296868191999437e+16, 'train_loss': 0.029269320644976272, 'epoch': 2.0})

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/distilbert_phishing_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/distilbert_phishing_model"
)

print("DistilBERT phishing model saved to Drive!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT phishing model saved to Drive!


### 5.3 DistilBERT Inference on Test Set

The saved fine-tuned model and tokenizer are reloaded from Google Drive. Predictions are generated on the test and evaluation sets using a `Trainer` instance configured for inference only.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_bert = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/distilbert_phishing_model"
)

print("Trained DistilBERT model loaded!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Trained DistilBERT model loaded!


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/distilbert_phishing_model"
)

print("Tokenizer loaded!")

Tokenizer loaded!


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
from transformers import Trainer

test_trainer = Trainer(
    model=model_bert,
    data_collator=data_collator
)

In [ ]:
test_predictions = test_trainer.predict(test_tokenized)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(test_predictions.predictions, axis=1)
y_true = test_predictions.label_ids

print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10870
           1       1.00      0.99      1.00      9434

    accuracy                           1.00     20304
   macro avg       1.00      1.00      1.00     20304
weighted avg       1.00      1.00      1.00     20304

Confusion Matrix:
[[10829    41]
 [   51  9383]]


In [ ]:
# DistilBERT inference on the evaluation set
distilbert_eval_predictions = test_trainer.predict(eval_tokenized)

print(
    "Evaluation predictions shape:",
    distilbert_eval_predictions.predictions.shape
)

Evaluation predictions shape: (20300, 2)


In [ ]:
import numpy as np

np.save(
    "/content/drive/MyDrive/distilbert_eval_logits.npy",
    distilbert_eval_predictions.predictions
)

np.save(
    "/content/drive/MyDrive/distilbert_eval_labels.npy",
    distilbert_eval_predictions.label_ids
)

print("DistilBERT evaluation predictions saved to Drive!")

DistilBERT evaluation predictions saved to Drive!


In [ ]:
import numpy as np

distilbert_eval_predictions = np.load(
    "/content/drive/MyDrive/distilbert_eval_logits.npy"
)

distilbert_eval_labels = np.load(
    "/content/drive/MyDrive/distilbert_eval_labels.npy"
)

print("Evaluation logits shape:", distilbert_eval_predictions.shape)
print("Evaluation labels shape:", distilbert_eval_labels.shape)

Evaluation logits shape: (20300, 2)
Evaluation labels shape: (20300,)


In [ ]:
import numpy as np

np.save(
    "/content/drive/MyDrive/distilbert_test_predictions.npy",
    test_predictions.predictions
)

np.save(
    "/content/drive/MyDrive/distilbert_test_labels.npy",
    test_predictions.label_ids
)

print("Test predictions and labels saved!")

Test predictions and labels saved!


In [ ]:
import numpy as np

test_predictions = np.load(
    "/content/drive/MyDrive/distilbert_test_predictions.npy"
)

y_true = np.load(
    "/content/drive/MyDrive/distilbert_test_labels.npy"
)

y_pred = np.argmax(test_predictions, axis=1)

print("Predictions:", test_predictions.shape)
print("Labels:", y_true.shape)

Predictions: (20304, 2)
Labels: (20304,)


In [ ]:
print(type(test_predictions))
print(test_predictions.shape)
print(test_predictions.shape)

<class 'numpy.ndarray'>
(20304, 2)
(20304, 2)


In [ ]:
import pandas as pd

df_test = pd.read_parquet(
    "/content/drive/MyDrive/phishing_test_clean.parquet"
)

print("Shape:", df_test.shape)
print("Columns:", df_test.columns.tolist())

Shape: (20304, 5)
Columns: ['text', 'subject', 'label', 'sender', 'email']


### 5.4 DistilBERT Error Analysis

False positives (legitimate emails misclassified as phishing) and false negatives (phishing emails classified as legitimate) are identified and examined.

In [ ]:
error_df = df_test.copy()

error_df["true_label"] = y_true
error_df["predicted_label"] = y_pred

# False positives: Legitimate → Phishing
false_positives = error_df[
    (error_df["true_label"] == 0) &
    (error_df["predicted_label"] == 1)
]

# False negatives: Phishing → Legitimate
false_negatives = error_df[
    (error_df["true_label"] == 1) &
    (error_df["predicted_label"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

False Positives: 41
False Negatives: 51


In [ ]:
print("===== FALSE POSITIVES =====")
display(false_positives[["email", "true_label", "predicted_label"]].head(10))

print("\n===== FALSE NEGATIVES =====")
display(false_negatives[["email", "true_label", "predicted_label"]].head(10))

===== FALSE POSITIVES =====


,email,true_label,predicted_label
316,Receive full credit towards a Masters degree f...,0,1
687,Complimentary creative software from Avery Ave...,0,1
1294,Ado <taux3@central.susx.ac.uk> Could you ple...,0,1
1674,Documentation Specialist Seeking Contract Work...,0,1
2035,Girlfriend's Day Di Cutsforth <dicutsforth@att...,0,1
2167,International Romanian Summer Course in Transy...,0,1
2382,Cross the finish line faster with Our Personal...,0,1
3219,"Sun, Dec 30 @ The Gatsby - Pre New Years 2002 ...",0,1
3267,Fountainhead; XXI Black International Cinema B...,0,1
4873,=?iso-8859-1?Q?Matrox_Parhelia=99_now_availabl...,0,1



===== FALSE NEGATIVES =====


,email,true_label,predicted_label
223,Last Minute Jamaica Travel Specials changes1@e...,1,0
1511,Essential Chinese Phrases for a Successful Tri...,1,0
1524,FOX Sports Net to Broadcast UCLA/USC Game FOXS...,1,0
1754,Dominate Your Fantasy Football League with Hud...,1,0
2255,"European ""Community Patent"" Proposal Derailed ...",1,0
2600,There was no way to help the horse out. fearfu...,1,0
2808,Job Offer Mark Maxwell <markmaxwell@myway.com>...,1,0
2854,[141.217.1.231])<br> &nbsp;&nbsp;&nbsp;&nbsp...,1,0
3128,"Do we have a Synergy? Hema@Izhuta.com Hi Rick,...",1,0
3462,please update the cwnframework . org listing b...,1,0


In [ ]:
# Add email length
error_df["email_length"] = error_df["email"].astype(str).str.len()

# Separate correct and incorrect predictions
false_positives = error_df[
    (error_df["true_label"] == 0) &
    (error_df["predicted_label"] == 1)
]

false_negatives = error_df[
    (error_df["true_label"] == 1) &
    (error_df["predicted_label"] == 0)
]

print("FALSE POSITIVES")
print("Count:", len(false_positives))
print("Average email length:",
      round(false_positives["email_length"].mean(), 2))
print("Median email length:",
      false_positives["email_length"].median())

print("\nFALSE NEGATIVES")
print("Count:", len(false_negatives))
print("Average email length:",
      round(false_negatives["email_length"].mean(), 2))
print("Median email length:",
      false_negatives["email_length"].median())

FALSE POSITIVES
Count: 41
Average email length: 5109.12
Median email length: 1279.0

FALSE NEGATIVES
Count: 51
Average email length: 1823.47
Median email length: 1187.0


In [ ]:
from scipy.special import softmax

# Convert DistilBERT logits into probabilities
probabilities = softmax(predictions, axis=1)

# Confidence = probability assigned to the predicted class
confidence = probabilities.max(axis=1)

error_df["confidence"] = confidence

false_positives = error_df[
    (error_df["true_label"] == 0) &
    (error_df["predicted_label"] == 1)
]

false_negatives = error_df[
    (error_df["true_label"] == 1) &
    (error_df["predicted_label"] == 0)
]

print("FALSE POSITIVES")
print("Average confidence:",
      round(false_positives["confidence"].mean(), 4))
print("Minimum confidence:",
      round(false_positives["confidence"].min(), 4))
print("Maximum confidence:",
      round(false_positives["confidence"].max(), 4))

print("\nFALSE NEGATIVES")
print("Average confidence:",
      round(false_negatives["confidence"].mean(), 4))
print("Minimum confidence:",
      round(false_negatives["confidence"].min(), 4))
print("Maximum confidence:",
      round(false_negatives["confidence"].max(), 4))

FALSE POSITIVES
Average confidence: 0.9265
Minimum confidence: 0.514
Maximum confidence: 0.9999

FALSE NEGATIVES
Average confidence: 0.9585
Minimum confidence: 0.7008
Maximum confidence: 0.9997


In [ ]:
print("===== MOST CONFIDENT FALSE POSITIVES =====")

display(
    false_positives
    .sort_values("confidence", ascending=False)
    [["email", "true_label", "predicted_label", "confidence"]]
    .head(10)
)

print("\n===== MOST CONFIDENT FALSE NEGATIVES =====")

display(
    false_negatives
    .sort_values("confidence", ascending=False)
    [["email", "true_label", "predicted_label", "confidence"]]
    .head(10)
)

===== MOST CONFIDENT FALSE POSITIVES =====


,email,true_label,predicted_label,confidence
15073,adres marysi robertlu @ friko 6 . onet . pl,0,1,0.999913
2035,Girlfriend's Day Di Cutsforth <dicutsforth@att...,0,1,0.999751
9175,"Iris Nutter dawndi <Dawndi@uswest.net> Hey, ch...",0,1,0.999680
1674,Documentation Specialist Seeking Contract Work...,0,1,0.999512
15459,Cultivating Poverty in India Normand Arsenault...,0,1,0.999436
8643,"Special Offer - CD/DVD Replication Services ""V...",0,1,0.998921
6511,Benelli Sport 10610 Long Island Shooting Cente...,0,1,0.998766
5214,"g . mulder , indirecte en directe directieven ...",0,1,0.998663
8586,"--Breaking News from The Evil Gerald-- ""The Ev...",0,1,0.998473
3267,Fountainhead; XXI Black International Cinema B...,0,1,0.997254



===== MOST CONFIDENT FALSE NEGATIVES =====


,email,true_label,predicted_label,confidence
7502,Yiwu Building Materials Trade Fair 2008 Jingu ...,1,0,0.999661
14045,New Post/Thread Notification: Private Investit...,1,0,0.999661
4241,This is a Bad Idea SAS INSTALL <null@mtu.edu> ...,1,0,0.999632
19704,Yahoo Broadcast Services as option for Corpora...,1,0,0.999602
9393,"Veronica Tapia <vtapia@mail.giga.com> Hello, ...",1,0,0.999366
2854,[141.217.1.231])<br> &nbsp;&nbsp;&nbsp;&nbsp...,1,0,0.999158
17487,"Cost Saving Proposal ""Barlota, Prameesh"" <pram...",1,0,0.999065
11690,FWD: Requested documents David Saunders <david...,1,0,0.999027
9859,"review any book , pc or mac software pgm , con...",1,0,0.998561
9439,6:30am Howard asked the guys if they wanted to...,1,0,0.998438


---

## 6. Model Evaluation and Comparison

All three models (Logistic Regression, LSTM, DistilBERT) are evaluated on the test set. Accuracy, precision, recall, and F1 are reported side by side.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Predictions from all three models
# y_test = true labels

results = []

# Logistic Regression
results.append({
    "Model": "TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_test),
    "Precision": precision_score(y_test, y_pred_test),
    "Recall": recall_score(y_test, y_pred_test),
    "F1": f1_score(y_test, y_pred_test)
})

# LSTM
results.append({
    "Model": "LSTM",
    "Accuracy": accuracy_score(y_test, y_pred_lstm),
    "Precision": precision_score(y_test, y_pred_lstm),
    "Recall": recall_score(y_test, y_pred_lstm),
    "F1": f1_score(y_test, y_pred_lstm)
})

# DistilBERT
y_pred_bert = np.argmax(test_predictions, axis=1)

results.append({
    "Model": "DistilBERT",
    "Accuracy": accuracy_score(y_test, y_pred_bert),
    "Precision": precision_score(y_test, y_pred_bert),
    "Recall": recall_score(y_test, y_pred_bert),
    "F1": f1_score(y_test, y_pred_bert)
})

comparison_df = pd.DataFrame(results)

print("FINAL MODEL COMPARISON")
display(comparison_df.round(4))

FINAL MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + Logistic Regression,0.9840,0.9802,0.9855,0.9828
1,LSTM,0.9764,0.9768,0.9723,0.9746
2,DistilBERT,0.9955,0.9956,0.9946,0.9951


In [ ]:
print(type(test_predictions))
print(test_predictions.shape)

<class 'numpy.ndarray'>
(20304, 2)


### 6.1 DistilBERT Standalone Inference Function

A `predict_email` function is defined that loads the saved DistilBERT model and tokenizer, runs inference on a single email string, and returns the predicted label and confidence.

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/distilbert_phishing_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert.to(device)
model_bert.eval()

print("DistilBERT model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

DistilBERT model loaded successfully!
Device: cpu


In [ ]:
def predict_email(email_text):

    inputs = tokenizer(
        email_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model_bert(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()

    # Based on our dataset:
    # 0 = legitimate
    # 1 = phishing

    label = "PHISHING" if predicted_class == 1 else "LEGITIMATE"

    return label, confidence

In [ ]:
email = """
Congratulations! You have won a $1,000 Amazon gift card.

Click the link below to claim your prize immediately.
Your account will be suspended if you do not verify your information
within 24 hours.

Click here: http://example.com/verify
"""

prediction, confidence = predict_email(email)

print("Prediction:", prediction)
print("Confidence:", f"{confidence * 100:.2f}%")

Prediction: PHISHING
Confidence: 99.99%


In [ ]:
email = """
Hi John,

Just wanted to confirm that our meeting has been moved to
3 PM tomorrow.

Please let me know if that works for you.

Best,
David
"""

prediction, confidence = predict_email(email)

print("Prediction:", prediction)
print("Confidence:", f"{confidence * 100:.2f}%")

Prediction: LEGITIMATE
Confidence: 99.99%


---

## 7. Confidence-Based Cascade Design

This section develops the **LR → DistilBERT cascade**: Logistic Regression handles all predictions first; only emails where LR predicts *legitimate* with low confidence are escalated to DistilBERT. Several cascade strategies are evaluated on the evaluation set.

In [ ]:
# ACTUAL DEPLOYEMENT STARTS
# Get Logistic Regression probabilities on evaluation data

lr_eval_probs = model.predict_proba(X_eval)[:, 1]

print("Evaluation samples:", len(lr_eval_probs))
print("Minimum probability:", lr_eval_probs.min())
print("Maximum probability:", lr_eval_probs.max())

Evaluation samples: 20300
Minimum probability: 8.875149874752995e-07
Maximum probability: 0.9999561717540024


In [ ]:
import numpy as np
lr_eval_confidence = np.maximum(
    lr_eval_probs,
    1 - lr_eval_probs
)

print("Minimum confidence:", lr_eval_confidence.min())
print("Maximum confidence:", lr_eval_confidence.max())

Minimum confidence: 0.5010870842603191
Maximum confidence: 0.9999991124850125


In [ ]:
thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

for threshold in thresholds:

    confident = lr_eval_confidence >= threshold
    escalated = ~confident

    print(
        f"Threshold {threshold:.2f} | "
        f"LR handles: {confident.mean()*100:.2f}% | "
        f"DistilBERT receives: {escalated.mean()*100:.2f}%"
    )

Threshold 0.50 | LR handles: 100.00% | DistilBERT receives: 0.00%
Threshold 0.60 | LR handles: 98.72% | DistilBERT receives: 1.28%
Threshold 0.70 | LR handles: 97.10% | DistilBERT receives: 2.90%
Threshold 0.80 | LR handles: 94.38% | DistilBERT receives: 5.62%
Threshold 0.90 | LR handles: 87.89% | DistilBERT receives: 12.11%
Threshold 0.95 | LR handles: 78.57% | DistilBERT receives: 21.43%
Threshold 0.99 | LR handles: 45.76% | DistilBERT receives: 54.24%


In [ ]:
print(df_eval.columns.tolist())

['text', 'subject', 'label', 'sender', 'email']


In [ ]:
print([name for name in list(globals().keys()) if "pred" in name.lower()])

['y_pred', 'predict_email', 'distilbert_eval_predictions', 'test_predictions', 'bert_eval_pred']


In [ ]:
print("predictions type:", type(test_predictions))
print("predictions shape:", test_predictions.shape)
print("predictions labels:", test_predictions.shape)

predictions type: <class 'numpy.ndarray'>
predictions shape: (20304, 2)
predictions labels: (20304, 2)


In [ ]:
from scipy.special import softmax

# Convert DistilBERT logits to probabilities
bert_eval_probs = softmax(
    distilbert_eval_predictions,
    axis=1
)

# Convert probabilities to class predictions
bert_eval_pred = np.argmax(
    bert_eval_probs,
    axis=1
)

print("DistilBERT evaluation predictions:", len(bert_eval_pred))

DistilBERT evaluation predictions: 20300


In [ ]:
import numpy as np

# Logistic Regression predictions on evaluation set
lr_eval_pred = (lr_eval_probs >= 0.5).astype(int)

print("LR evaluation predictions:", len(lr_eval_pred))

LR evaluation predictions: 20300


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_eval = df_eval["label"].values

thresholds = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99]

cascade_results = []

for threshold in thresholds:

    # Emails LR is confident about
    confident = lr_eval_confidence >= threshold

    # Start with LR predictions
    cascade_pred = lr_eval_pred.copy()

    # Replace uncertain LR predictions with DistilBERT predictions
    cascade_pred[~confident] = bert_eval_pred[~confident]

    cascade_results.append({
        "Threshold": threshold,
        "LR Usage %": confident.mean() * 100,
        "DistilBERT Usage %": (~confident).mean() * 100,
        "Accuracy": accuracy_score(y_eval, cascade_pred),
        "Precision": precision_score(y_eval, cascade_pred),
        "Recall": recall_score(y_eval, cascade_pred),
        "F1": f1_score(y_eval, cascade_pred)
    })

cascade_df = pd.DataFrame(cascade_results)

display(cascade_df.round(4))

NameError: name 'lr_eval_pred' is not defined

In [ ]:
y_eval = df_eval["label"].values

In [ ]:
import numpy as np
import pandas as pd

# LR predicted class and confidence
lr_eval_pred = np.argmax(
    np.column_stack([
        1 - lr_eval_probs,
        lr_eval_probs
    ]),
    axis=1
)

lr_eval_confidence = np.maximum(
    lr_eval_probs,
    1 - lr_eval_probs
)

# DistilBERT predicted class
bert_eval_pred = np.argmax(
    distilbert_eval_predictions,
    axis=1
)

threshold_analysis = []

for threshold in [0.96, 0.97, 0.98, 0.99]:

    # Emails that LR considers uncertain
    uncertain = lr_eval_confidence < threshold

    # Among uncertain emails, which LR predictions are wrong?
    lr_wrong = (lr_eval_pred != y_eval) & uncertain

    # Among those LR errors, which ones does DistilBERT get correct?
    bert_corrects_lr_errors = (
        (bert_eval_pred == y_eval) & lr_wrong
    )

    # Final cascade prediction
    cascade_pred = lr_eval_pred.copy()
    cascade_pred[uncertain] = bert_eval_pred[uncertain]

    # Metrics
    lr_errors = lr_wrong.sum()
    bert_corrections = bert_corrects_lr_errors.sum()

    recovery_rate = (
        bert_corrections / lr_errors * 100
        if lr_errors > 0 else 0
    )

    remaining_errors = (
        cascade_pred != y_eval
    ).sum()

    threshold_analysis.append({
        "Threshold": threshold,
        "LR Uncertain Emails": uncertain.sum(),
        "LR Errors in Uncertain": lr_errors,
        "DistilBERT Corrects LR Errors": bert_corrections,
        "Error Recovery %": recovery_rate,
        "Remaining Cascade Errors": remaining_errors
    })

threshold_analysis_df = pd.DataFrame(threshold_analysis)

display(threshold_analysis_df.round(2))

,Threshold,LR Uncertain Emails,LR Errors in Uncertain,DistilBERT Corrects LR Errors,Error Recovery %,Remaining Cascade Errors
0,0.96,5122,314,268,85.35,107
1,0.97,6170,319,271,84.95,107
2,0.98,7863,322,273,84.78,106
3,0.99,11010,326,275,84.36,105


In [ ]:
from sklearn.metrics import confusion_matrix

# All LR errors on evaluation set
lr_errors_mask = lr_eval_pred != y_eval

# False Positives: LR predicted 1, actual 0
fp_mask = (lr_eval_pred == 1) & (y_eval == 0)

# False Negatives: LR predicted 0, actual 1
fn_mask = (lr_eval_pred == 0) & (y_eval == 1)

print("Total LR Errors:", lr_errors_mask.sum())
print("False Positives:", fp_mask.sum())
print("False Negatives:", fn_mask.sum())

# High-confidence LR errors
high_conf_error = lr_errors_mask & (lr_eval_confidence >= 0.96)

print("\nHigh-confidence LR errors (>= 0.96):", high_conf_error.sum())

# Breakdown
high_conf_fp = fp_mask & (lr_eval_confidence >= 0.96)
high_conf_fn = fn_mask & (lr_eval_confidence >= 0.96)

print("High-confidence False Positives:", high_conf_fp.sum())
print("High-confidence False Negatives:", high_conf_fn.sum())

Total LR Errors: 326
False Positives: 192
False Negatives: 134

High-confidence LR errors (>= 0.96): 12
High-confidence False Positives: 9
High-confidence False Negatives: 3


In [ ]:
# important for not sending a legit mail to phishing class.

# FP-focused analysis of Logistic Regression

# Legitimate emails that LR incorrectly predicts as phishing
fp_mask = (y_eval == 0) & (lr_eval_pred == 1)

# Among those FPs, how confident was LR?
fp_confidences = lr_eval_confidence[fp_mask]

print("Total False Positives:", fp_mask.sum())
print("Average FP confidence:", fp_confidences.mean())
print("Minimum FP confidence:", fp_confidences.min())
print("Maximum FP confidence:", fp_confidences.max())

print("\nFPs by confidence range:")

ranges = [
    (0.50, 0.60),
    (0.60, 0.70),
    (0.70, 0.80),
    (0.80, 0.90),
    (0.90, 0.95),
    (0.95, 0.96),
    (0.96, 0.97),
    (0.97, 0.98),
    (0.98, 0.99),
    (0.99, 1.00)
]

for low, high in ranges:
    count = ((fp_confidences >= low) &
             (fp_confidences < high)).sum()

    print(f"{low:.2f}–{high:.2f}: {count}")

Total False Positives: 192
Average FP confidence: 0.6875602032503081
Minimum FP confidence: 0.5018333461510547
Maximum FP confidence: 0.9894854865033573

FPs by confidence range:
0.50–0.60: 65
0.60–0.70: 47
0.70–0.80: 31
0.80–0.90: 35
0.90–0.95: 5
0.95–0.96: 0
0.96–0.97: 3
0.97–0.98: 3
0.98–0.99: 3
0.99–1.00: 0


In [ ]:
# Evaluation labels
y_eval = df_eval["label"].values

# LR probability + prediction
lr_eval_pred = (lr_eval_probs >= 0.5).astype(int)
lr_eval_confidence = np.maximum(lr_eval_probs, 1 - lr_eval_probs)

# DistilBERT prediction
bert_eval_pred = np.argmax(distilbert_eval_predictions, axis=1)

### 7.1 Cascade Decision Logic

In [ ]:
#Logic
#LR → PHISHING: final = phishing, regardless of confidence.
#LR → LEGITIMATE + confidence ≥ threshold: final = legitimate.
#LR → LEGITIMATE + confidence < threshold: send to DistilBERT.

### 7.2 Strategy C — Security-First Cascade

**Goal:** Minimise false negatives (phishing emails that slip through as legitimate).  
LR flags all phishing predictions as final. Only LR-legitimate predictions with confidence below the threshold are escalated to DistilBERT.

In [ ]:
# ============================================================
# STRATEGY C — SECURITY-FIRST CASCADE
# Goal: Minimize False Negatives
# ============================================================

THRESHOLD = 0.99

# Start with LR predictions
strategy_c_pred = lr_eval_pred.copy()

# We only send LR's LEGITIMATE predictions to DistilBERT
lr_legitimate = (lr_eval_pred == 0)

# Among LR-legitimate predictions, find the uncertain ones
needs_bert = (
    lr_legitimate &
    (lr_eval_confidence < THRESHOLD)
)

# DistilBERT gets ONLY these uncertain legitimate emails
strategy_c_pred[needs_bert] = bert_eval_pred[needs_bert]


# ------------------------------------------------------------
# Evaluate final cascade
# ------------------------------------------------------------

cm = confusion_matrix(y_eval, strategy_c_pred)

print("STRATEGY C — SECURITY-FIRST CASCADE")
print("-----------------------------------")

print("Threshold:", THRESHOLD)

print(
    f"DistilBERT usage: "
    f"{needs_bert.mean() * 100:.2f}%"
)

print(
    "Accuracy :",
    accuracy_score(y_eval, strategy_c_pred)
)

print(
    "Precision:",
    precision_score(y_eval, strategy_c_pred)
)

print(
    "Recall   :",
    recall_score(y_eval, strategy_c_pred)
)

print(
    "F1       :",
    f1_score(y_eval, strategy_c_pred)
)

print("\nConfusion Matrix:")
print(cm)

print("\nFalse Positives:", cm[0, 1])
print("False Negatives:", cm[1, 0])

STRATEGY C — SECURITY-FIRST CASCADE
-----------------------------------
Threshold: 0.99
DistilBERT usage: 27.34%
Accuracy : 0.9885221674876847
Precision: 0.9784666597316134
Recall   : 0.9972434266327396
F1       : 0.9877658177999475

Confusion Matrix:
[[10661   207]
 [   26  9406]]

False Positives: 207
False Negatives: 26


### 7.3 Strategy D — Selective Second Opinion

Low-confidence LR predictions of *either* class are escalated to DistilBERT.

In [ ]:
# ============================================================
# STRATEGY D — SELECTIVE SECOND OPINION
# Send ONLY low-confidence LR predictions to DistilBERT
# ============================================================

THRESHOLD = 0.99

# Start with LR's prediction
strategy_d_pred = lr_eval_pred.copy()

# Any LR prediction below the confidence threshold
# gets a second opinion from DistilBERT
needs_bert = lr_eval_confidence < THRESHOLD

# Replace LR's prediction with BERT's prediction
# ONLY for these uncertain emails
strategy_d_pred[needs_bert] = bert_eval_pred[needs_bert]


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

cm = confusion_matrix(y_eval, strategy_d_pred)

print("STRATEGY D — SELECTIVE SECOND OPINION")
print("--------------------------------------")

print("Threshold:", THRESHOLD)

print(
    f"DistilBERT usage: "
    f"{needs_bert.mean() * 100:.2f}%"
)

print(
    "Accuracy :",
    accuracy_score(y_eval, strategy_d_pred)
)

print(
    "Precision:",
    precision_score(y_eval, strategy_d_pred)
)

print(
    "Recall   :",
    recall_score(y_eval, strategy_d_pred)
)

print(
    "F1       :",
    f1_score(y_eval, strategy_d_pred)
)

print("\nConfusion Matrix:")
print(cm)

print("\nFalse Positives:", cm[0, 1])
print("False Negatives:", cm[1, 0])

STRATEGY D — SELECTIVE SECOND OPINION
--------------------------------------
Threshold: 0.99
DistilBERT usage: 54.24%
Accuracy : 0.9948275862068966
Precision: 0.9957478473477198
Recall   : 0.993108566581849
F1       : 0.994426455756675

Confusion Matrix:
[[10828    40]
 [   65  9367]]

False Positives: 40
False Negatives: 65


### 7.4 Strategy E — Asymmetric Cascade for Minimising False Negatives

Separate thresholds are applied for phishing and legitimate LR predictions.

In [ ]:
# ============================================================
# STRATEGY E — ASYMMETRIC CASCADE FOR MINIMIZING FN
# ============================================================

PHISHING_THRESHOLD = 0.95
LEGITIMATE_THRESHOLD = 0.99

# Start with LR predictions
strategy_e_pred = lr_eval_pred.copy()

# LR predicted phishing
lr_phishing = (lr_eval_pred == 1)

# LR predicted legitimate
lr_legitimate = (lr_eval_pred == 0)

# Uncertain phishing predictions → BERT
uncertain_phishing = (
    lr_phishing &
    (lr_eval_confidence < PHISHING_THRESHOLD)
)

# Uncertain legitimate predictions → BERT
uncertain_legitimate = (
    lr_legitimate &
    (lr_eval_confidence < LEGITIMATE_THRESHOLD)
)

# All emails requiring BERT
needs_bert = uncertain_phishing | uncertain_legitimate

# BERT gets the final decision for these emails
strategy_e_pred[needs_bert] = bert_eval_pred[needs_bert]


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

cm = confusion_matrix(y_eval, strategy_e_pred)

print("STRATEGY E — FN-OPTIMIZED ASYMMETRIC CASCADE")
print("---------------------------------------------")

print("Phishing threshold  :", PHISHING_THRESHOLD)
print("Legitimate threshold:", LEGITIMATE_THRESHOLD)

print(f"\nTotal DistilBERT usage: {needs_bert.mean() * 100:.2f}%")
print(f"Phishing → BERT      : {uncertain_phishing.sum()}")
print(f"Legitimate → BERT    : {uncertain_legitimate.sum()}")

print("\nAccuracy :", accuracy_score(y_eval, strategy_e_pred))
print("Precision:", precision_score(y_eval, strategy_e_pred))
print("Recall   :", recall_score(y_eval, strategy_e_pred))
print("F1       :", f1_score(y_eval, strategy_e_pred))

print("\nConfusion Matrix:")
print(cm)

print("\nFalse Positives:", cm[0, 1])
print("False Negatives:", cm[1, 0])

STRATEGY E — FN-OPTIMIZED ASYMMETRIC CASCADE
---------------------------------------------
Phishing threshold  : 0.95
Legitimate threshold: 0.99

Total DistilBERT usage: 37.62%
Phishing → BERT      : 2085
Legitimate → BERT    : 5551

Accuracy : 0.9947290640394089
Precision: 0.9951152171604545
Recall   : 0.9935326547921968
F1       : 0.994323306276195

Confusion Matrix:
[[10822    46]
 [   61  9371]]

False Positives: 46
False Negatives: 61


### 7.5 Holdout Test Evaluation — Strategy C + Quarantine

> **Note:** The cell below specifies the final holdout evaluation logic for Strategy C on the test set. It requires `test_predictions` (DistilBERT test predictions) and Logistic Regression test predictions (`lr_test_pred`, `lr_test_confidence`). To generate them when running on a fresh environment: `lr_test_probs = model.predict_proba(X_test)[:, 1]`, `lr_test_pred = (lr_test_probs >= 0.5).astype(int)`, and `lr_test_confidence = np.maximum(lr_test_probs, 1 - lr_test_probs)`.


In [ ]:
# ============================================================
# FINAL TEST — STRATEGY C + QUARANTINE
# ============================================================

TEST_THRESHOLD = 0.95

# Start with LR predictions
final_test_pred = lr_test_pred.copy()

# Only LEGITIMATE predictions with confidence < 95%
# are sent to DistilBERT
test_needs_bert = (
    (lr_test_pred == 0) &
    (lr_test_confidence < TEST_THRESHOLD)
)

# DistilBERT gives the final decision for these emails
final_test_pred[test_needs_bert] = np.argmax(
    test_predictions[test_needs_bert],
    axis=1
)

# ============================================================
# FINAL TEST METRICS
# ============================================================

cm = confusion_matrix(y_test, final_test_pred)

print("FINAL TEST RESULTS — STRATEGY C + QUARANTINE")
print("==============================================")

print(f"Threshold: {TEST_THRESHOLD}")
print(f"DistilBERT usage: {test_needs_bert.mean() * 100:.2f}%")

print("\nAccuracy :", accuracy_score(y_test, final_test_pred))
print("Precision:", precision_score(y_test, final_test_pred))
print("Recall   :", recall_score(y_test, final_test_pred))
print("F1       :", f1_score(y_test, final_test_pred))

print("\nConfusion Matrix:")
print(cm)

print("\nFalse Positives (Legitimate → Phishing):", cm[0, 1])
print("False Negatives (Phishing → Legitimate):", cm[1, 0])

NameError: name 'lr_test_pred' is not defined

---

## 8. Saving Final Models

The trained Logistic Regression model and the fine-tuned DistilBERT model and tokenizer are persisted to Google Drive for use in inference.

In [ ]:
import joblib

joblib.dump(
    model,
    "/content/drive/MyDrive/phishing_lr_model.pkl"
)

print("Final Logistic Regression model saved to Drive!")

In [ ]:
print("Possible DistilBERT / tokenizer variables:")

for name in list(globals().keys()):
    if any(x in name.lower() for x in ["bert", "tokenizer", "model", "trainer"]):
        print(name)

In [ ]:
# ============================================================
# SAVE FINAL DISTILBERT MODEL + TOKENIZER TO GOOGLE DRIVE
# ============================================================

BERT_SAVE_PATH = "/content/drive/MyDrive/phishing_distilbert"

# Save model
model_bert.save_pretrained(BERT_SAVE_PATH)

# Save tokenizer
tokenizer.save_pretrained(BERT_SAVE_PATH)

print("DistilBERT model and tokenizer saved to Drive!")
print(f"Location: {BERT_SAVE_PATH}")

---

## 9. Full Inference Pipeline with Quarantine

The final LR + DistilBERT cascade pipeline is assembled. Emails that remain uncertain after both models are written to a JSON-based **quarantine log** rather than being automatically classified.

In [ ]:
# ============================================================
# LOAD FINAL MODELS
# ============================================================

import joblib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# TF-IDF
tfidf = joblib.load(
    "/content/drive/MyDrive/tfidf_vectorizer.pkl"
)

# Logistic Regression
lr_model = joblib.load(
    "/content/drive/MyDrive/phishing_lr_model.pkl"
)

# DistilBERT
BERT_PATH = "/content/drive/MyDrive/phishing_distilbert"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_PATH)

bert_model.eval()

print("All final models loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

All final models loaded successfully!


In [ ]:
# ============================================================
# FINAL PHISHING EMAIL PREDICTOR
# STRATEGY C + QUARANTINE
# ============================================================

THRESHOLD = 0.95


def predict_email(email_text):

    # --------------------------------------------------------
    # 1. Logistic Regression
    # --------------------------------------------------------

    email_tfidf = tfidf.transform([email_text])

    lr_prob = lr_model.predict_proba(email_tfidf)[0, 1]

    lr_pred = int(lr_prob >= 0.5)

    lr_confidence = max(lr_prob, 1 - lr_prob)


    # --------------------------------------------------------
    # 2. Strategy C routing
    # --------------------------------------------------------

    # IMPORTANT:
    # If LR predicts phishing → DON'T send to BERT.
    # It goes directly to quarantine.

    if lr_pred == 1:

        return {
            "prediction": "PHISHING",
            "action": "QUARANTINE",
            "model_used": "Logistic Regression",
            "lr_confidence": lr_confidence
        }


    # --------------------------------------------------------
    # 3. Uncertain legitimate prediction → DistilBERT
    # --------------------------------------------------------

    if lr_confidence < THRESHOLD:

        inputs = bert_tokenizer(
            email_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = bert_model(**inputs)

        bert_probs = torch.softmax(
            outputs.logits,
            dim=1
        )[0]

        bert_pred = int(torch.argmax(bert_probs))

        bert_confidence = float(
            bert_probs[bert_pred]
        )

        if bert_pred == 1:
            action = "QUARANTINE"
            prediction = "PHISHING"
        else:
            action = "DELIVER"
            prediction = "LEGITIMATE"

        return {
            "prediction": prediction,
            "action": action,
            "model_used": "DistilBERT",
            "lr_confidence": lr_confidence,
            "bert_confidence": bert_confidence
        }


    # --------------------------------------------------------
    # 4. Confident legitimate → deliver
    # --------------------------------------------------------

    return {
        "prediction": "LEGITIMATE",
        "action": "DELIVER",
        "model_used": "Logistic Regression",
        "lr_confidence": lr_confidence
    }

In [ ]:
# ============================================================
# TEST THE COMPLETE PHISHING DETECTION PIPELINE
# ============================================================

test_emails = {

    "Legitimate 1": """
    Hi John,

    Just a reminder that our project meeting is scheduled for
    tomorrow at 10 AM. Please bring the latest project report.

    Thanks,
    Sarah
    """,

    "Legitimate 2": """
    Your monthly electricity bill is now available in your account.
    You can view and download the bill from the official customer portal.

    Regards,
    Customer Support
    """,

    "Phishing 1": """
    URGENT: Your bank account will be suspended today.
    Verify your account immediately by clicking the link below
    and entering your username, password and OTP.
    """,

    "Phishing 2": """
    Congratulations! You have won a $5,000 reward.
    To claim your prize, confirm your banking information
    within the next 24 hours using the link below.
    """
}


for name, email in test_emails.items():

    result = predict_email(email)

    print("=" * 60)
    print(name)
    print("=" * 60)

    for key, value in result.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    print()

Legitimate 1
prediction: LEGITIMATE
action: DELIVER
model_used: Logistic Regression
lr_confidence: 0.9952

Legitimate 2
prediction: PHISHING
action: QUARANTINE
model_used: Logistic Regression
lr_confidence: 0.7565

Phishing 1
prediction: PHISHING
action: QUARANTINE
model_used: Logistic Regression
lr_confidence: 0.9748

Phishing 2
prediction: PHISHING
action: QUARANTINE
model_used: Logistic Regression
lr_confidence: 0.9658



In [ ]:
# Find an evaluation email where Strategy C sends LR's
# legitimate + low-confidence prediction to DistilBERT

bert_indices = np.where(
    (lr_eval_pred == 0) &
    (lr_eval_confidence < 0.95)
)[0]

print("Emails that trigger BERT:", len(bert_indices))

# Take the first one
idx = bert_indices[0]

print("\nOriginal email:\n")
print(df_eval.iloc[idx]["email"])

print("\nTrue label:", y_eval[idx])
print("LR prediction:", lr_eval_pred[idx])
print("LR confidence:", lr_eval_confidence[idx])
print("DistilBERT prediction:", bert_eval_pred[idx])

Emails that trigger BERT: 2265

Original email:

[EDIS]  THIS TSUNAMI INFORMATION BULLETIN IS FOR ALASKA BRITISH [Urgent: Statewide] EDIS Email Service <edismail@incident.com> WEST COAST AND ALASKA TSUNAMI WARNING CENTER/NOAA/NWS ISSUED 03/31/2002 AT 0723 UTC ...THIS TSUNAMI INFORMATION BULLETIN IS FOR ALASKA, BRITISH    COLUMBIA, WASHINGTON, OREGON, AND CALIFORNIA ONLY... NO - REPEAT NO - WATCH OR WARNING IS IN EFFECT. EARTHQUAKE DATA:  PRELIMINARY MAGNITUDE: 7.3  LOCATION: 24.7N 122.2E - TAIWAN REGION                     TIME: 2153 AST 03/30/2002        2253 PST 03/30/2002        0653 UTC 03/31/2002 EVALUATION:  BASED ON LOCATION AND MAGNITUDE THE EARTHQUAKE WAS NOT  SUFFICIENT TO GENERATE A TSUNAMI DAMAGING TO CALIFORNIA,  OREGON, WASHINGTON, BRITISH COLUMBIA OR ALASKA. SOME AREAS MAY  EXPERIENCE SMALL SEA LEVEL CHANGES. IN AREAS OF INTENSE  SHAKING, LOCALLY GENERATED TSUNAMIS CAN BE TRIGGERED BY SLUMPING. THE PACIFIC TSUNAMI WARNING CENTER WILL ISSUE TSUNAMI BULLETINS FOR HAWAII AN

In [ ]:
#CREATING QUARANTINE FOR NOW
import os
import json
from datetime import datetime

QUARANTINE_DIR = "/content/drive/MyDrive/phishing_quarantine"

os.makedirs(QUARANTINE_DIR, exist_ok=True)

print("Quarantine directory ready:")
print(QUARANTINE_DIR)

Quarantine directory ready:
/content/drive/MyDrive/phishing_quarantine


In [ ]:
def quarantine_email(email_text, prediction_result):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    filename = f"quarantine_{timestamp}.json"
    filepath = os.path.join(QUARANTINE_DIR, filename)

    quarantine_record = {
        "timestamp": datetime.now().isoformat(),
        "email": email_text,
        "prediction": prediction_result["prediction"],
        "action": "QUARANTINE",
        "model_used": prediction_result["model_used"],
        "lr_confidence": prediction_result.get("lr_confidence"),
        "bert_confidence": prediction_result.get("bert_confidence")
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(quarantine_record, f, indent=4)

    return filepath

In [ ]:
def process_email(email_text):

    result = predict_email(email_text)

    if result["action"] == "QUARANTINE":

        filepath = quarantine_email(
            email_text,
            result
        )

        result["quarantine_file"] = filepath

    return result

In [ ]:
test_email = """
URGENT: Your bank account has been compromised.

Verify your account immediately by clicking the link below
and entering your username, password and OTP.

Failure to verify within 24 hours will result in account suspension.
"""

result = process_email(test_email)

print(result)

{'prediction': 'PHISHING', 'action': 'QUARANTINE', 'model_used': 'Logistic Regression', 'lr_confidence': np.float64(0.9771611962080576), 'quarantine_file': '/content/drive/MyDrive/phishing_quarantine/quarantine_20260901_055404_695288.json'}


In [ ]:
# ============================================================
# CLEAN INFERENCE API
# Strategy C + Quarantine
# ============================================================

def analyze_email(email_text):

    result = process_email(email_text)

    response = {
        "prediction": result["prediction"],
        "action": result["action"],
        "model_used": result["model_used"],
        "lr_confidence": round(result["lr_confidence"] * 100, 2)
    }

    # Add BERT confidence only when BERT was used
    if "bert_confidence" in result:
        response["bert_confidence"] = round(
            result["bert_confidence"] * 100, 2
        )

    # Add quarantine location if applicable
    if "quarantine_file" in result:
        response["quarantine_file"] = result["quarantine_file"]

    return response

In [ ]:
email = """
URGENT: Your account has been suspended.

Please verify your identity immediately by clicking
the link below and providing your account credentials.
Failure to verify within 24 hours will result in
permanent account closure.
"""

result = analyze_email(email)

for key, value in result.items():
    print(f"{key}: {value}")

prediction: PHISHING
action: QUARANTINE
model_used: Logistic Regression
lr_confidence: 98.66
quarantine_file: /content/drive/MyDrive/phishing_quarantine/quarantine_20260901_055404_722986.json


---

## 10. FastAPI REST API

The full cascade pipeline is wrapped in a FastAPI application. A `/predict` endpoint accepts an email string and returns the classification result, model used, confidence scores, and quarantine file path (if applicable).

In [ ]:
!pip install fastapi uvicorn nest-asyncio

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Phishing Email Detection API",
    description="LR + DistilBERT cascade for phishing detection",
    version="1.0"
)


class EmailRequest(BaseModel):
    email: str


@app.post("/predict")
def predict(request: EmailRequest):

    result = analyze_email(request.email)

    return result

In [ ]:
 # ============================================================
# TEST FASTAPI ENDPOINT LOCALLY
# ============================================================

from fastapi.testclient import TestClient

client = TestClient(app)

response = client.post(
    "/predict",
    json={
        "email": """
        URGENT: Your bank account has been suspended.
        Click the link immediately to verify your account
        and prevent permanent closure.
        """
    }
)

print("Status code:", response.status_code)
print("\nAPI response:")

print(response.json())

Status code: 200

API response:
{'prediction': 'PHISHING', 'action': 'QUARANTINE', 'model_used': 'Logistic Regression', 'lr_confidence': 99.34, 'quarantine_file': '/content/drive/MyDrive/phishing_quarantine/quarantine_20260901_055412_469679.json'}
